# Week 3 — Cleaning your data
## group2a · Clinic visits

**Your question**
> Which departments have the longest patient wait times, and how does that vary by visit type and over time?

**What this notebook does.** Pulls the raw data out of the database, fixes the
problems you found in week 2, and writes clean tables back into your own
schema. Power BI reads those tables in week 4.

**How to use it.** Every section has an explanation, then a cell to run, then
a `TODO` where you make a decision. The decisions are the work — the code
around them is scaffolding so you are not starting from a blank page.

**Before you start:** have your week 2 `data_quality_notes.md` open. Every
number you wrote there tells you what to fix here.

---
### One rule
Run this notebook top to bottom, in order. If it only works when you run cells
out of sequence, it is not finished — someone else in your group has to be able
to run it from scratch and get the same tables.


## 1. Setup

Run this once per session. Colab forgets everything when it disconnects, so
you will run it again tomorrow.


In [1]:
!pip install -q psycopg2-binary sqlalchemy

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from getpass import getpass

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print("pandas", pd.__version__)



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


pandas 2.2.2


### Connect

`getpass` hides your password as you type it, so it never ends up saved in the
notebook. **Never type your password directly into a cell** — the notebook goes
to GitHub and the password would go with it.


In [2]:
HOST   = "internship-db.coh86gwewtxb.us-east-1.rds.amazonaws.com"
DB     = "internship"
USER   = "group2a"
SCHEMA_RAW   = "raw_clinic"
SCHEMA_MINE  = "group2a"

password = getpass()

engine = create_engine(
    f"postgresql+psycopg2://{USER}:{password}@{HOST}:5432/{DB}?sslmode=require"
)

# quick check
pd.read_sql(f"SELECT count(*) AS rows FROM {SCHEMA_RAW}.visits", engine)


,rows
0,25100


You should see **25,100**. If not, stop — something is
wrong with the connection, not with your code.


## 2. Load the raw tables

Pull all four into pandas. They are small enough to hold in memory
comfortably.


In [3]:
df = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.visits", engine)
patients = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.patients", engine)
doctors = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.doctors", engine)
departments = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.departments", engine)

print('visits   ', df.shape)
print('patients'.ljust(12), patients.shape)
print('doctors'.ljust(12), doctors.shape)
print('departments'.ljust(12), departments.shape)


visits    (25100, 8)
patients     (900, 6)
doctors      (40, 4)
departments  (8, 3)


`.shape` gives (rows, columns). Check these against what you
recorded in week 2 — if a number is different, find out why before going on.


In [4]:
df.head(10)


,visit_id,visit_date,patient_id,doctor_id,department_id,visit_type,wait_minutes,consultation_fee
0,6884,2025-11-21,238,39,7,Follow-up,148,50.0
1,21688,2025-04-03,121,31,2,Outpatient,84,0.0
2,17110,2024/04/06,820,23,1,Outpatient,172,80.0
3,14652,2025-03-17,794,15,1,Follow-up,73,35.0
4,16764,06-Oct-2024,105,1,3,Outpatient,7,20.0
5,11529,2024-05-03,384,2,8,Outpatient,129,0.0
6,13951,2025-12-15,186,2,5,None,188,20.0
7,1950,2024/09/05,111,30,3,Follow-up,94,120.0
8,11016,2024/09/18,870,1,6,Outpatient,102,0.0
9,19108,2025-10-12,354,5,4,Follow-up,46,20.0


In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   visit_id          25100 non-null  int64  
 1   visit_date        25100 non-null  object 
 2   patient_id        25100 non-null  int64  
 3   doctor_id         25100 non-null  int64  
 4   department_id     25100 non-null  int64  
 5   visit_type        23592 non-null  object 
 6   wait_minutes      25100 non-null  int64  
 7   consultation_fee  25100 non-null  float64
dtypes: float64(1), int64(5), object(2)
memory usage: 1.5+ MB


Look at `df.info()` carefully. Note which columns pandas
thinks are `object` — that means text. The date column will be one of them,
which is the whole problem.


## 3. Record where you are starting

Before changing anything, capture the numbers. At the end you will compare
against these and prove the cleaning worked.


In [6]:
before = {
    'rows':       len(df),
    'duplicates': len(df) - df['visit_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['visit_type'].nunique(),
}
before


{'rows': 25100, 'duplicates': 100, 'missing': 1508, 'categories': 12}

---
## 4. Remove duplicate rows

Week 2 told you how many exact duplicates there are. `drop_duplicates()`
removes them, keeping the first occurrence.


In [7]:
# 1. How many rows share a visit_id with another row?
dup_ids = df[df.duplicated(subset='visit_id', keep=False)]
print("rows involved in duplicated visit_ids:", len(dup_ids))
print("extra rows (visit_id level):", df.duplicated(subset='visit_id').sum())

# 2. How many are exact copies across ALL columns?
print("extra rows (exact full-row):", df.duplicated().sum())

rows involved in duplicated visit_ids: 200
extra rows (visit_id level): 100
extra rows (exact full-row): 100


In [8]:
print("before:", len(df))
df = df.drop_duplicates()
print("after: ", len(df))
print("removed:", before['rows'] - len(df))


before: 25100
after:  25000
removed: 100


**TODO — write down the number removed. Does it match your
week 2 figure?** 100 was removed and it matches week 2 figure.

If it does not, you are looking at something different from what you counted.
Work out which before continuing.


---
## 5. Standardise the messy categories

This is the one that would silently split your totals in Power BI. `visit_type`
has the same values written several ways — different capitalisation, stray
spaces.

`.str.strip()` removes leading and trailing spaces. `.str.title()` makes it
Title Case. Pick one form and apply it everywhere.


In [9]:
# what it looks like now
df['visit_type'].value_counts(dropna=False)


visit_type
Outpatient       11313
Follow-up         6212
Emergency         3188
None              1500
  Outpatient       523
OUTPATIENT         511
outpatient         490
FOLLOW-UP          284
  Follow-up        283
follow-up          271
emergency          150
  Emergency        144
EMERGENCY          131
Name: count, dtype: int64

In [10]:
df['visit_type'] = df['visit_type'].str.strip().str.title()

df['visit_type'].value_counts(dropna=False)


visit_type
Outpatient    12837
Follow-Up      7050
Emergency      3613
None           1500
Name: count, dtype: int64

**TODO — how many categories now, and how many before?**

Before: 12 distinct values (13 groups if we count the blanks).

Now: 3 real categories (Outpatient, Follow-Up, Emergency), plus the missing rows still showing as None.

Now do the same for the other text columns. Week 2 should have told you which
ones are affected — it is not only this one.


In [11]:
#Checking the patients table for distinct values in 'sex' and 'region' columns:
for col in ['sex', 'region']:
    print(col, "| before:", patients[col].nunique(), "distinct values")

sex | before: 8 distinct values
region | before: 31 distinct values


In [12]:
# Text columns in patients: standardize casing and whitespace
# Problem: the same value was stored several ways (e.g. 'FEMALE', 'female', '  Female ')
patients['sex'] = patients['sex'].str.strip().str.title()
patients['region'] = patients['region'].str.strip().str.title()

# check your work
patients['sex'].value_counts(dropna=False).head(15)

sex
Female    464
Male      436
Name: count, dtype: int64

---
## 6. Parse the dates

`visit_date` is text, in four different formats. `pd.to_datetime` with
`format='mixed'` handles them, and `dayfirst=True` tells it to read `03/04/2025`
as 3 April rather than 4 March.

**This is a real decision, not a setting.** Nothing in the data proves which
reading is right. Whatever you choose, write it down in your cleaning notes and
be ready to defend it.



In [13]:
# what formats are present
df['visit_date'].str.len().value_counts()


visit_date
10    23532
11     1468
Name: count, dtype: int64

In [14]:
df['visit_date'] = pd.to_datetime(
    df['visit_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'      # anything unparseable becomes NaT rather than crashing
)

print("could not parse:", df['visit_date'].isna().sum())
print("range:", df['visit_date'].min(), "to", df['visit_date'].max())


could not parse: 0
range: 2024-01-01 00:00:00 to 2025-12-30 00:00:00


In [15]:
with engine.connect() as conn:
    raw_dates = pd.read_sql("SELECT visit_id, visit_date AS raw_text FROM raw_clinic.visits", conn)

check = raw_dates.head(10).copy()
check['parsed'] = pd.to_datetime(check['raw_text'], format='mixed', dayfirst=True)
check['parsed_display'] = check['parsed'].dt.strftime('%d %b %Y')   # e.g. 04 Mar 2024, unambiguous

print(check[['visit_id', 'raw_text', 'parsed', 'parsed_display']])

   visit_id     raw_text     parsed parsed_display
0      6884   2025-11-21 2025-11-21    21 Nov 2025
1     21688   2025-04-03 2025-04-03    03 Apr 2025
2     17110   2024/04/06 2024-04-06    06 Apr 2024
3     14652   2025-03-17 2025-03-17    17 Mar 2025
4     16764  06-Oct-2024 2024-10-06    06 Oct 2024
5     11529   2024-05-03 2024-05-03    03 May 2024
6     13951   2025-12-15 2025-12-15    15 Dec 2025
7      1950   2024/09/05 2024-09-05    05 Sep 2024
8     11016   2024/09/18 2024-09-18    18 Sep 2024
9     19108   2025-10-12 2025-10-12    12 Oct 2025


In [16]:
#Backwards compactibility check: 
with engine.connect() as conn:
    raw = pd.read_sql("SELECT visit_id, visit_date FROM raw_clinic.visits", conn).drop_duplicates()

day_first  = pd.to_datetime(raw['visit_date'], format='mixed', dayfirst=True,  errors='coerce')
month_first = pd.to_datetime(raw['visit_date'], format='mixed', dayfirst=False, errors='coerce')

differs = day_first != month_first
print("rows with a different date:", differs.sum(), "of", len(raw))
print("share:", round(differs.sum() / len(raw) * 100, 1), "%")
print("range day-first:  ", day_first.min(), "to", day_first.max())
print("range month-first:", month_first.min(), "to", month_first.max())
print(raw.assign(day_first=day_first, month_first=month_first)[differs].head(10))

rows with a different date: 535 of 25000
share: 2.1 %
range day-first:   2024-01-01 00:00:00 to 2025-12-30 00:00:00
range month-first: 2024-01-01 00:00:00 to 2025-12-30 00:00:00
     visit_id  visit_date  day_first month_first
10       3656  09/11/2024 2024-11-09  2024-09-11
89      12440  03/05/2025 2025-05-03  2025-03-05
109      3163  02/10/2025 2025-10-02  2025-02-10
120     18654  06/07/2024 2024-07-06  2024-06-07
129     10200  01/05/2024 2024-05-01  2024-01-05
136       818  06/08/2024 2024-08-06  2024-06-08
166     16612  09/02/2024 2024-02-09  2024-09-02
220      2308  09/07/2024 2024-07-09  2024-09-07
223     23316  06/03/2025 2025-03-06  2025-06-03
238     14755  01/12/2025 2025-12-01  2025-01-12


**TODO — does that date range make sense now?**

Compare it with what the text version gave you in week 2. This is where the
text-sorting problem finally goes away.

If any rows failed to parse, decide what to do with them and say why.

   
   
    Yes, the date range now makes sense. After parsing, visit_date runs from 1 January 2024 to 30 December 2025, two full calendar years, with nothing before 2024 or in the future. All 25,000 rows parsed (0 failures), so no rows needed dropping or fixing.
    In week 2 the text version gave 01/01/2024 to 31-Oct-2025. That result was wrong because text sorts character by character, not by date, so the "latest" date came out two months earlier than the real one. Now that the column is a real date type, sorting is chronological and the problem is gone.




---
## 7. Deal with impossible values

Week 2 found negative wait times. Look at them before deciding.


In [17]:
bad = df[df['wait_minutes'] < 0]
print("rows affected:", len(bad))
bad.head(10)


rows affected: 50


,visit_id,visit_date,patient_id,doctor_id,department_id,visit_type,wait_minutes,consultation_fee
619,17062,2025-02-08,216,24,3,Emergency,-118,0.0
629,10785,2025-05-25,356,27,7,Outpatient,-34,50.0
654,5577,2025-02-20,477,14,3,Outpatient,-130,50.0
692,19342,2024-02-03,69,32,4,Follow-Up,-198,0.0
975,7396,2024-08-08,642,35,4,Follow-Up,-46,0.0
1620,19437,2025-10-15,163,20,4,Emergency,-147,35.0
2696,4096,2024-07-12,484,30,8,Outpatient,-72,0.0
2997,11412,2025-08-03,700,36,1,Outpatient,-84,35.0
3204,6392,2025-01-22,213,3,5,Follow-Up,-151,20.0
3910,7275,2024-08-22,102,35,7,Outpatient,-66,50.0


**TODO — decide, and write down why.**

Three defensible options. There is no single right answer, but there is a wrong
one: doing it silently.

1. **Drop them.** Clean, but you lose whatever else was in those rows.
2. **Set them to NULL.** Keeps the row, marks the value as unknown.
3. **Fix them** — if a negative looks like a data-entry sign error, taking the
   absolute value may be justified. Only if you can argue it.


In [18]:
#checking if the negative wait times are due to data entry errors or if they are valid cases. 
neg = bad[bad['wait_minutes'] < 0]
print(neg['department_id'].value_counts())
print(neg['visit_type'].value_counts())
print(neg['visit_date'].dt.to_period('M').value_counts().sort_index())

department_id
1    9
5    9
2    9
7    7
4    7
3    5
8    3
6    1
Name: count, dtype: int64
visit_type
Outpatient    25
Follow-Up     19
Emergency      4
Name: count, dtype: int64
visit_date
2024-01    2
2024-02    3
2024-03    2
2024-05    2
2024-06    2
2024-07    3
2024-08    6
2024-09    2
2024-11    3
2024-12    1
2025-01    5
2025-02    3
2025-05    2
2025-06    2
2025-07    3
2025-08    1
2025-09    2
2025-10    1
2025-11    2
2025-12    3
Freq: M, Name: count, dtype: int64


In [19]:
# TODO: implement your decision. One of these, or your own.

# option 1 — drop
# df = df[~df.index.isin(bad.index)]

# option 2 — set to NULL
df.loc[bad.index, 'wait_minutes'] = np.nan

print("rows now:", len(df))
print("wait_minutes now missing:", df['wait_minutes'].isna().sum())


rows now: 25000
wait_minutes now missing: 50


---
## 8. Deal with orphan keys

Some `patient_id` values in your fact table point at
`patients` records that do not exist. A plain join would drop these rows
silently — which is exactly why you are handling them deliberately.


In [20]:
valid = set(patients['patient_id'])
orphans = df[~df['patient_id'].isin(valid)]

print("orphan rows:", len(orphans))
orphans[['visit_id', 'patient_id']].head(10)


orphan rows: 62


,visit_id,patient_id
23,5999,90042
30,17251,90000
1029,6489,90023
1506,12104,90019
2248,1840,90020
2446,21597,90039
2510,51,90012
3711,7197,90030
3789,6073,90002
3920,17899,90027


**TODO — decide, and write down why.**

1. **Drop them.** Simple, and you lose real transactions.
2. **Keep them, pointing at an "Unknown" record.** Preserves the totals, and
   your dashboard shows an Unknown category — which is honest.

Check every foreign key, not just this one.


In [21]:
# TODO: implement your decision

# option 1 — drop
# df = df[df['patient_id'].isin(valid)]

# option 2 — add an Unknown row to the dimension, then repoint orphans at it
unknown = pd.DataFrame([{'patient_id': -1}])
patients = pd.concat([patients, unknown], ignore_index=True)
df.loc[~df['patient_id'].isin(valid), 'patient_id'] = -1

# label the text columns of the Unknown record so Power BI doesn't show blanks
for col in patients.select_dtypes(include='object').columns:
    patients.loc[patients['patient_id'] == -1, col] = 'Unknown'
    
print("rows now:", len(df)) 
print("patients rows:", len(patients))                                    
print("visits pointing at Unknown:", (df['patient_id'] == -1).sum())   
print("orphans left:", (~df['patient_id'].isin(patients['patient_id'])).sum())  


rows now: 25000
patients rows: 901
visits pointing at Unknown: 62
orphans left: 0


---
## 9. Handle missing values

Decide **per column**. A NULL is not always a mistake — sometimes it means
something real, and filling it in would be inventing data.


In [22]:
df.isna().sum().sort_values(ascending=False)


visit_type          1500
wait_minutes          50
visit_id               0
visit_date             0
patient_id             0
doctor_id              0
department_id          0
consultation_fee       0
dtype: int64

**TODO — for each column with missing values, decide and record:**

| Column | How many | Decision | Why |
|---|---|---|---|
| `visit_type` | 1500| Fill them with 'Unknown'|This is a category we group by. The visits are real and their wait times are valid, so we have kept them and they will show as their own "Unknown" group in Power BI. |
| | | | |

Options: leave as NULL (honest, Power BI shows blanks), fill with a label like
`'Unknown'` (good for text you will group by), or drop the row (only if the row
is useless without it).


In [23]:
#Filling in missing values for visit_type with unknown

# example — label missing text so it groups properly in Power BI
df['visit_type'] = df['visit_type'].fillna('Unknown')

df.isna().sum().sort_values(ascending=False).head()


wait_minutes    50
visit_id         0
visit_date       0
patient_id       0
doctor_id        0
dtype: int64

---
## 10. Prove it worked

Re-run your week 2 checks on the cleaned data. Every problem should now be
gone or accounted for.


In [24]:
after = {
    'rows':       len(df),
    'duplicates': len(df) - df['visit_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['visit_type'].nunique(),
}

pd.DataFrame([before, after], index=['before', 'after'])


,rows,duplicates,missing,categories
before,25100,100,1508,12
after,25000,0,50,4


**TODO — explain every number that changed.**

If rows went down, you should be able to say exactly how many were duplicates,
how many were impossible values, and how many were orphans. If the numbers do
not add up, something happened that you did not intend.

    Rows went from 25,100 to 25,000, a drop of exactly 100. All 100 were duplicate rows (exact copies sharing a visit_id). No rows were lost to impossible values or orphan keys: the 50 negative wait times were set to NULL and the 62 orphan patient_id values were repointed to an Unknown record, so those rows stayed. Missing values fell from 1,508 to 50, but the two figures count different columns. Missing visit_type fell from 1,508 to 1,500 when the duplicates were removed (8 duplicate copies had a blank), then to 0 when we filled the blanks with 'Unknown'. The 50 that remain are wait_minutes values we set to NULL on purpose because they were negative. The categories in visit_type fell from 12 spelling variants to 4: Outpatient, Follow-Up, Emergency and Unknown.

---
# &#9888;&#65039; STOP — read this before running anything below

Everything so far happened **in memory**, on your own copy. Nothing you have
run has changed the database. That is why every person in the group can run
sections 1 to 10 safely and should.

What comes next is different. It writes to the database, and everyone in your
group shares one login and one schema.

---

### Sections 1&ndash;10 &nbsp;&middot;&nbsp; EVERYONE

Run them. Argue about the TODOs. Make your own mistakes — nothing you do here
can affect anyone else.

### Section 11 &nbsp;&middot;&nbsp; ONE PERSON ONLY

The write-back. Agree your decisions as a group first, then **one person** runs
it — normally the week 3 phase lead.

### Section 12 &nbsp;&middot;&nbsp; OPTIONAL, INDIVIDUAL

Want to try a different cleaning decision without destroying the group's
version? There is a safe way to do it. See section 12.

---

**Why it matters.** `if_exists='replace'` rebuilds the table every run, so
nothing duplicates. But **the last person to run wins**. If two of you made
different decisions, the table holds whichever ran last — while your written
notes describe the other one. In week 6 your dashboard totals will not match
your documentation and you will hunt a bug that is not a bug.


---
## 11. Write the clean tables back &nbsp;&middot;&nbsp; ONE PERSON ONLY

> **Before running this cell, check with your group.** Has everyone agreed the
> decisions in sections 7, 8 and 9? Is it clear who is running it?

Into **your own schema**, not the raw one. `if_exists='replace'` rebuilds each
table from scratch — use `'append'` by mistake and running twice silently
doubles your data.


In [25]:
patients = patients.drop_duplicates(subset='patient_id', keep='first').reset_index(drop=True)

print("patients rows:", len(patients))                              # 901
print("Unknown rows:", (patients['patient_id'] == -1).sum())        # 1
print("duplicate ids:", patients['patient_id'].duplicated().sum())  # 0
print("visits at Unknown:", (df['patient_id'] == -1).sum())         # 62
print("orphans left:", (~df['patient_id'].isin(patients['patient_id'])).sum())  # 0

patients rows: 901
Unknown rows: 1
duplicate ids: 0
visits at Unknown: 62
orphans left: 0


In [26]:
# ONE PERSON ONLY. Check with your group before running.

# chunksize + method='multi' batches the inserts. Without them
# pandas sends one row at a time, which is slow over the internet.
df.to_sql('visits_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False,
          chunksize=1000, method='multi')
patients.to_sql('patients_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False,
          chunksize=1000, method='multi')
doctors.to_sql('doctors_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False,
          chunksize=1000, method='multi')
departments.to_sql('departments_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False,
          chunksize=1000, method='multi')

print("written")


written


### Confirm they landed

Count each table you just wrote and check against what you expect.


In [27]:
for t in ['visits_clean', 'patients_clean', 'doctors_clean', 'departments_clean']:
    n = pd.read_sql(f"SELECT count(*) AS n FROM {SCHEMA_MINE}.{t}", engine)
    print(t.ljust(22), int(n.iloc[0,0]))


visits_clean           25000
patients_clean         901
doctors_clean          40
departments_clean      8


In [28]:
with engine.connect() as conn:
    cols = pd.read_sql("""
        SELECT column_name FROM information_schema.columns
        WHERE table_schema = 'group2a' AND table_name = 'visits_clean'
        ORDER BY ordinal_position
    """, conn)
    n = pd.read_sql("SELECT count(*) AS n FROM group2a.visits_clean", conn)['n'][0]

print(cols['column_name'].tolist())
print("rows:", n)

['visit_id', 'visit_date', 'patient_id', 'doctor_id', 'department_id', 'visit_type', 'wait_minutes', 'consultation_fee']
rows: 25000


---
## 12. Optional &nbsp;&middot;&nbsp; your own experiments

Anyone may create their own tables in `group2a`. Nothing stops you. That is
useful — you can try a different decision and compare it against the group's
version without destroying anything.

**One rule: prefix it `zz_` and your name.** It sorts to the bottom of the
table list and it is obvious to everyone that it is temporary.


In [29]:
# safe — your own table, does not touch the group's
MY_NAME = "Regbert"      # <-- change this

df.to_sql(f"zz_{MY_NAME}_visits", engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False,
          chunksize=1000, method='multi')

print("wrote", f"{SCHEMA_MINE}.zz_{MY_NAME}_visits")


wrote group2a.zz_Regbert_visits


### What is in the schema right now?

Run this any time. Four `_clean` tables is healthy. If there are eleven, people
have been experimenting and not cleaning up.


In [30]:
pd.read_sql(f"""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = '{SCHEMA_MINE}'
    ORDER BY table_name
""", engine)


,table_name
0,departments_clean
1,doctors_clean
2,patients_clean
3,test
4,visits_clean
5,zz_Regbert_visits


### Clean up after yourself

When you are done comparing, drop your own tables. Leaving them makes the
schema unreadable by week 6, and a tidy schema is part of what gets
assessed.


In [31]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {SCHEMA_MINE}.zz_{MY_NAME}_visits"))

print("dropped")


dropped


> **Never drop a table that is not yours.** A `DROP TABLE` on
> `visits_clean` breaks everyone's work, including the Power BI file in
> week 4, and there is no undo.


---
## Before you finish week 3

- [ ] This notebook runs top to bottom without errors, from a fresh runtime
- [ ] Someone else in the group has run it and got the same tables
- [ ] Every TODO above has a written answer
- [ ] Your cleaning decisions and reasons are in your week 3 form
- [ ] This notebook is committed to `notebooks/` in your repository
- [ ] Your password is **not** anywhere in the notebook

**Test it properly:** Runtime → Restart runtime, then Run all. If it fails, it
is not finished.

Next week you connect Power BI to `group2a` and build the model on these tables.
